# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) with the `mlcroissant` library following the Croissant schema best practices.

### Dataset Source
The dataset is described via a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and data records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")

## 2. Data Overview
List available record sets, their `@id`s, and their fields. This allows understanding of the data organization before extracting actual records.

**Note:** In this notebook we reference all record sets and fields by their `@id` as required by FAIR and Croissant best practices.

In [ ]:
# Show all record sets and fields using their @id
print("Available record sets and their fields:\n")
recordsets_dict = {}
for rs in dataset.record_sets:
    print(f"Record set: {rs['@id']}")
    field_ids = [f['@id'] for f in rs.get('field', [])]
    print(f"  Fields: {field_ids}\n")
    recordsets_dict[rs['@id']] = field_ids
# For short-hand use, collect all recordset @ids
recordset_ids = list(recordsets_dict.keys())

## 3. Data Extraction
Load data from all available record sets into DataFrames keyed by record set `@id`. Use these for inspection and downstream processing.

In [ ]:
dataframes = {}

for record_set_id in recordset_ids:
    print(f"Loading records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(2), "\n")

# For subsequent steps, pick the main protein record set, which for this dataset is likely:
# '@id': 'https://api.app.sen.science/frontiers/7154140/8875020f-242a-4f80-afd4-a4cbac8715f0/rs:mainTable'
# If the main table is named differently, select the most extensive record set.

# Try to select a record set ID corresponding to main quantitative results
main_record_set_id = None
for rid in recordset_ids:
    if 'mainTable' in rid or 'Main' in rid or 'protein' in rid:
        main_record_set_id = rid
        break
# Fallback: choose first record set
if main_record_set_id is None and len(recordset_ids) > 0:
    main_record_set_id = recordset_ids[0]

print(f"Selected for EDA: record set '@id': {main_record_set_id}")
print(f"Columns in selected record set: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set, performing example data filtering, normalization, and grouping.

- All references are by `@id`.
- Numeric and grouping fields are chosen based on available columns in the selected record set.

In [ ]:
# Inspect column names to select numeric and grouping fields by their @id
main_df = dataframes[main_record_set_id]
print("Data columns:", main_df.columns.tolist())

# Attempt to select a numeric field (e.g. coverage, MW, peptide_count)
# We'll select the first suitable field found.
numeric_field = None
possible_numeric_fields = [c for c in main_df.columns if any(x in c.lower() for x in ['coverage', 'mw', 'count', 'abundance', 'intensity'])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # Fallback: first numeric-type column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break

# Show unique values and pick group field
group_field = None
possible_categorical_fields = [c for c in main_df.columns if any(x in c.lower() for x in ['group', 'category', 'sample', 'type'])]
if possible_categorical_fields:
    group_field = possible_categorical_fields[0]
else:
    # Fallback: None
    group_field = None

if numeric_field is not None:
    print(f"Selected numeric field for filtering and normalization: '{numeric_field}'")
    # Ensure column is numeric
    main_df[numeric_field] = pd.to_numeric(main_df[numeric_field], errors='coerce')
    threshold = main_df[numeric_field].mean()
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): n={len(filtered_df)}")

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None:
        print(f"Grouping by '{group_field}' ...")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Let's visualize the distribution of the numeric field used above, and its relation to any grouping field (if present).

**Note**: All visualizations are optional and depend on the specific fields present in the chosen record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(7,4))
    main_df[numeric_field].hist(bins=40)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # Boxplot by group field, if available
    if group_field is not None and group_field in main_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: No numeric field.")

## 6. Conclusion
In this notebook, we:
- Loaded the Mass Spectrometry dataset using the Croissant schema and `mlcroissant` library,
- Inspected available record sets and fields via their `@id`,
- Loaded main quantitative results into a DataFrame for EDA,
- Performed filtering and normalization on a numeric protein descriptor field,
- Optionally grouped data by experimental sample or group (if available),
- Visualized numeric field distributions.

This approach enables robust, schema-driven FAIR data analysis for multi-table, well-annotated datasets.